# Pelajaran 05 - Agentic RAG


## Setup

Notebook ini mendemonstrasikan pola Agentic RAG (Retrieval-Augmented Generation) menggunakan Microsoft Agent Framework.

**Prasyarat:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — endpoint layanan Azure AI Search Anda
- `AZURE_SEARCH_API_KEY` — kunci API Azure AI Search Anda
- Deployment Azure OpenAI yang dikonfigurasi melalui variabel lingkungan
- Azure CLI sudah terautentikasi (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Apa itu Agentic RAG?

RAG tradisional mengikuti alur tetap: mengambil dokumen, kemudian menghasilkan respons. **Agentic RAG** melangkah lebih jauh dengan memberikan agen otonomi untuk memutuskan **kapan** dan **bagaimana** mengambil informasi.

Dengan Agentic RAG, agen dapat:
- **Memutuskan** apakah pengambilan informasi diperlukan sebelum menjawab pertanyaan
- **Memilih** sumber data atau alat mana yang akan digunakan
- **Mengevaluasi** hasil yang diambil dan melakukan pengambilan lanjutan jika upaya pertama tidak memadai
- **Menggabungkan** informasi dari beberapa langkah pengambilan menjadi jawaban yang koheren

Ini membuat agen lebih fleksibel dan akurat dibandingkan dengan alur tetap ambil lalu buat respons.


## Membuat Alat Pencarian

Dalam Agentic RAG, sumber data eksternal dibungkus sebagai **alat** yang dapat dipanggil oleh agen sesuai permintaan. Ini memungkinkan agen memperlakukan pengambilan data sebagai satu tindakan lain yang dapat dilakukan, bukan sebagai langkah wajib.

Di bawah ini kami mendefinisikan basis pengetahuan perjalanan dan mengeksposnya sebagai alat yang dapat dipanggil agen untuk mencari informasi tujuan.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Membangun Agen RAG

Sekarang kita membuat agen yang diperintahkan untuk **selalu mengambil informasi sebelum menjawab**. Agen menggunakan alat `search_travel_knowledge` untuk mendasarkan jawabannya pada basis pengetahuan daripada mengandalkan data pelatihan sendiri.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Pengambilan Iteratif — Pola Maker-Checker

Keuntungan utama dari Agentic RAG adalah **pengambilan iteratif**. Agen dapat melakukan beberapa putaran pencarian untuk memverifikasi, menyempurnakan, atau memperluas temuan awalnya — mirip dengan alur kerja "maker-checker":

1. **Langkah Maker**: Agen mengambil informasi awal dan membuat draf jawaban.
2. **Langkah Checker**: Agen melakukan pengambilan tambahan untuk memverifikasi detail atau mengisi kekurangan.

Di bawah ini, agen ditanyakan sebuah pertanyaan yang memerlukan perbandingan beberapa tujuan, mendorongnya untuk melakukan pencarian berkali-kali.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Ringkasan

Dalam pelajaran ini Anda belajar bagaimana membangun sistem **Agentic RAG** menggunakan Microsoft Agent Framework:

- **Agentic RAG** memungkinkan agen untuk secara mandiri memutuskan kapan mengambil informasi, menjadikan pengambilan bersifat dinamis daripada tetap.
- **Alat sebagai sumber data**: Basis pengetahuan eksternal (seperti Azure AI Search) dibungkus sebagai alat yang dapat dipanggil oleh agen.
- **Pengambilan secara iteratif**: Pola maker-checker memungkinkan agen melakukan beberapa putaran pengambilan — mencari, memverifikasi, dan menyempurnakan — sebelum menghasilkan jawaban akhir.

Dalam produksi, Anda akan mengganti `TRAVEL_KNOWLEDGE_BASE` di memori dengan indeks Azure AI Search nyata untuk menangani pengambilan dokumen perjalanan berskala besar.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berusaha untuk akurat, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sahih. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau kesalahan interpretasi yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
